In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/adiitya0929/TinySQL-125M-WikiSQL

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/adiitya0929/TinySQL-125M-WikiSQL)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="adiitya0929/TinySQL-125M-WikiSQL")

config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  500MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/193 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.27M [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
# pip install -U transformers accelerate
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("adiitya0929/TinySQL-125M-WikiSQL")
model = AutoModelForCausalLM.from_pretrained("adiitya0929/TinySQL-125M-WikiSQL", device_map="auto")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
import sqlite3
import pandas as pd

def text_to_sql(df, question):
    """Turn a table (pandas DataFrame) + a question into SQL, using the prompt format the model was trained on."""
    cols = ", ".join(f'"{c}" {"REAL" if pd.api.types.is_numeric_dtype(df[c]) else "TEXT"}' for c in df.columns)
    prompt = f"<schema>CREATE TABLE t ({cols});</schema>\n<question>{question}</question>\n<sql>"
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    out = model.generate(ids, max_new_tokens=64)          # greedy; stops at </sql>
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def run_sql(df, sql):
    """Run the SQL on the DataFrame. The table must be called t; text is compared case-insensitively."""
    con = sqlite3.connect(":memory:")
    cols = ", ".join(f'"{c}" {"REAL" if pd.api.types.is_numeric_dtype(df[c]) else "TEXT COLLATE NOCASE"}' for c in df.columns)
    con.execute(f"CREATE TABLE t ({cols})")
    df.to_sql("t", con, if_exists="append", index=False)
    return pd.read_sql_query(sql, con)


In [ ]:
df = pd.DataFrame({
    "Name": ["Asha", "Ravi", "Meera", "John", "Sara"],
    "Department": ["Sales", "Engineering", "Engineering", "Sales", "HR"],
    "City": ["Pune", "Bengaluru", "Pune", "Mumbai", "Pune"],
    "Salary": [52000, 91000, 87000, 61000, 48000],
    "Years": [3, 7, 5, 4, 2],
})
df


,Name,Department,City,Salary,Years
0,Asha,Sales,Pune,52000,3
1,Ravi,Engineering,Bengaluru,91000,7
2,Meera,Engineering,Pune,87000,5
3,John,Sales,Mumbai,61000,4
4,Sara,HR,Pune,48000,2


In [ ]:
for q in ["Which department does Meera work in?",
          "How many employees work in Pune?",
          "What is the highest salary in the Engineering department?",
          "Which employees have more than 4 years?"]:
    sql = text_to_sql(df, q)
    print(q, "\n  SQL:   ", sql, "\n  Answer:", run_sql(df, sql).values.tolist(), "\n")


Which department does Meera work in? 
  SQL:    SELECT "Department" FROM t WHERE "Name" = 'meera'; 
  Answer: [['Engineering']] 

How many employees work in Pune? 
  SQL:    SELECT COUNT("Salary") FROM t WHERE "City" = 'Pune'; 
  Answer: [[3]] 

What is the highest salary in the Engineering department? 
  SQL:    SELECT MAX("Salary") FROM t WHERE "Department" = 'engineering'; 
  Answer: [[91000.0]] 

Which employees have more than 4 years? 
  SQL:    SELECT "Name" FROM t WHERE "Years" > '4'; 
  Answer: [['Ravi'], ['Meera']] 

